# Dashboards and Monitoring with Grafana

**Course Modules:**
1. Understanding Advanced Dashboard Design
2. Exploring Data Visualization Techniques
3. Building a Complete Grafana Team Solution

---
## Module 1 — Understanding Advanced Dashboard Design

### Adding Encryption to a Grafana Server

Securing the Grafana server with TLS/HTTPS ensures data in transit is protected.  
Key steps typically involve:
- Obtaining or generating a TLS certificate (e.g. via Let's Encrypt or self-signed)
- Configuring `grafana.ini` with the cert and key paths
- Restarting the Grafana service

In [ ]:
# Example: relevant grafana.ini settings for HTTPS
grafana_ini_tls = """
[server]
protocol = https
cert_file = /etc/grafana/grafana.crt
cert_key  = /etc/grafana/grafana.key
http_port = 3000
"""
print(grafana_ini_tls)

### Identifying a Monitoring Event

A monitoring event is any observable change in system state that may require attention:
- Threshold breaches (CPU, memory, disk, latency)
- Service availability changes (up/down)
- Anomalous patterns in time series data

Grafana surfaces monitoring events via **alerts** and **dashboard annotations**, allowing teams to correlate events across panels.

---
## Module 2 — Exploring Data Visualization Techniques

### CloudWatch Provisioner

Grafana can provision data sources automatically using YAML configuration files placed in `/etc/grafana/provisioning/datasources/`.  
This is useful for infrastructure-as-code and repeatable deployments.

In [ ]:
cloudwatch_provisioner = """
apiVersion: 1
datasources:
  - name: AWS-CloudWatch
    type: cloudwatch
    access: proxy
    uid: cloudwatch_001
    editable: false
    jsonData:
      authType: keys
      defaultRegion: us-east-1
    secureJsonData:
      accessKey: YOUR_AWS_ACCESS_KEY_ID
      secretKey: YOUR_AWS_SECRET_ACCESS_KEY
"""
print(cloudwatch_provisioner)

**Key fields explained:**

| Field | Description |
|-------|-------------|
| `type` | Data source plugin type (`cloudwatch`) |
| `access` | `proxy` = Grafana backend queries CloudWatch; `direct` = browser queries directly |
| `uid` | Stable identifier used by dashboards to reference this data source |
| `editable` | When `false`, prevents UI edits to this provisioned source |
| `authType` | `keys` uses explicit access/secret keys; `default` uses IAM role |
| `defaultRegion` | AWS region for queries |

### Defining Manual Queries

Each visualization panel has a query editor suited to the connected data source.

**CloudWatch example metrics query:**
- Namespace: `AWS/EC2`
- Metric: `CPUUtilization`
- Dimensions: `InstanceId = i-xxxxxxxxxxxxxxxxx`
- Statistics: `Average`
- Period: `60` seconds

**PromQL example:**
```promql
100 - (avg by(instance) (rate(node_cpu_seconds_total{mode="idle"}[5m])) * 100)
```

In [ ]:
# Example PromQL queries for common infrastructure metrics

queries = {
    "CPU usage %": '100 - (avg by(instance)(rate(node_cpu_seconds_total{mode="idle"}[5m])) * 100)',
    "Memory used %": '100 * (1 - (node_memory_MemAvailable_bytes / node_memory_MemTotal_bytes))',
    "Disk used %": '100 - (node_filesystem_avail_bytes{mountpoint="/"} / node_filesystem_size_bytes{mountpoint="/"} * 100)',
    "Network receive (bytes/s)": 'rate(node_network_receive_bytes_total[5m])',
    "Network transmit (bytes/s)": 'rate(node_network_transmit_bytes_total[5m])',
}

for name, query in queries.items():
    print(f"{name}:\n  {query}\n")

### Designing the Perfect Dashboard

Key principles for effective Grafana dashboards:

| Principle | Guidance |
|-----------|----------|
| **Audience first** | Design for the viewer — ops vs. exec vs. dev |
| **One story per dashboard** | Avoid cramming unrelated metrics together |
| **Use the right panel type** | Time series for trends, stat for single values, table for details, gauge for thresholds |
| **Consistent time ranges** | All panels should share the dashboard-level time picker |
| **Template variables** | Use `$instance`, `$job`, etc. to make dashboards reusable across targets |
| **Annotations** | Mark deployments and incidents on graphs to explain spikes |
| **Alert thresholds visible** | Show alert thresholds as lines directly on the panel |
| **Rows and grouping** | Group related panels in collapsible rows |

**Common panel types:**
- `Time series` — line/bar chart over time
- `Stat` — single big number with optional sparkline
- `Gauge` — radial or bar gauge for current value vs. range
- `Bar chart` — categorical comparisons
- `Table` — tabular data with sorting and filtering
- `Logs` — log stream panel (pairs with Loki)
- `Heatmap` — density over time
- `Geomap` — geographic data

---
## Module 3 — Building a Complete Grafana Team Solution

### Working With Users and Teams

Grafana's access control is organized at three levels:

| Level | Description |
|-------|-------------|
| **Organization** | Top-level namespace; a Grafana instance can host multiple orgs |
| **Team** | A group of users within an org — used for sharing folders/dashboards |
| **User** | Individual account with an org-level role |

**Org roles:**

| Role | Permissions |
|------|-------------|
| `Viewer` | Can view dashboards and data |
| `Editor` | Can create and edit dashboards |
| `Admin` | Full org-level admin (users, data sources, settings) |
| `Server Admin` | Instance-level: manage all orgs and users |

**Dashboard folder permissions** can be set per team or user, overriding the org-level role for more granular access.

### Working With Alerts

Grafana Alerting (unified alerting) consists of:

| Component | Role |
|-----------|------|
| **Alert rule** | Query + condition that defines when to fire |
| **Contact point** | Where notifications are sent (email, Slack, PagerDuty, webhooks, …) |
| **Notification policy** | Routes alerts to the right contact point based on labels |
| **Silences** | Suppress alerts during known maintenance windows |
| **Alert groups** | Deduplication and grouping of related alerts |

In [ ]:
import json

# Example: Grafana API payload to create an alert rule
alert_rule = {
    "title": "High CPU Usage",
    "condition": "C",
    "data": [
        {
            "refId": "A",
            "datasourceUid": "prometheus",
            "model": {
                "expr": '100 - (avg by(instance)(rate(node_cpu_seconds_total{mode="idle"}[5m])) * 100)',
                "intervalMs": 1000,
                "maxDataPoints": 43200,
            }
        },
        {
            "refId": "C",
            "datasourceUid": "-100",
            "model": {
                "conditions": [{"evaluator": {"params": [80], "type": "gt"}, "query": {"params": ["A"]}, "reducer": {"type": "last"}, "type": "query"}],
                "type": "classic_conditions"
            }
        }
    ],
    "noDataState": "NoData",
    "execErrState": "Error",
    "for": "5m",
    "labels": {"severity": "warning"},
    "annotations": {"summary": "CPU usage exceeded 80% for 5 minutes"}
}

print("POST /api/v1/provisioning/alert-rules")
print(json.dumps(alert_rule, indent=2))

### Common Contact Point Types

| Type | Use case |
|------|----------|
| **Email** | Individual or distribution list notifications |
| **Slack** | Team channel notifications |
| **PagerDuty** | On-call escalation |
| **OpsGenie** | Incident management |
| **Webhook** | Generic HTTP POST to any endpoint |
| **Microsoft Teams** | Org using M365 |
| **Telegram** | Lightweight chat notifications |

---
## Course Summary

| Module | Key Takeaway |
|--------|--------------|
| **Advanced Dashboard Design** | Secure Grafana with TLS; identify monitoring events via alerts and annotations |
| **Data Visualization Techniques** | Provision data sources (e.g. CloudWatch) via YAML; write manual PromQL/CloudWatch queries; design effective dashboards |
| **Complete Team Solution** | Manage users, teams, and roles; configure unified alerting with contact points and notification policies |

---
## Quick Reference

| Resource | URL / Path |
|----------|------------|
| Grafana TLS config | https://grafana.com/docs/grafana/latest/setup-grafana/configure-security/configure-security/ |
| Provisioning data sources | https://grafana.com/docs/grafana/latest/administration/provisioning/#data-sources |
| CloudWatch data source | https://grafana.com/docs/grafana/latest/datasources/aws-cloudwatch/ |
| Alert rules API | https://grafana.com/docs/grafana/latest/developers/http_api/alerting_provisioning/ |
| Unified alerting | https://grafana.com/docs/grafana/latest/alerting/ |
| Contact points | https://grafana.com/docs/grafana/latest/alerting/contact-points/ |
| Dashboard best practices | https://grafana.com/docs/grafana/latest/best-practices/best-practices-for-creating-dashboards/ |
| Provisioning dir | `/etc/grafana/provisioning/datasources/` |